# Prompt U-Net Complexity Analysis

This notebook measures the computational complexity, inference speed, and GPU memory usage of the **Prompt U-Net** model.

To produce numbers **directly comparable** with the UniverSeg benchmark:
- Uses `float32` precision (not `mixed_float16`)
- Uses `@tf.function`-compiled inference (eliminates Python/eager-mode overhead)
- Simulates a full volume of **128 × 128×128 2D slices**
- Synchronises GPU after every slice (equivalent to `torch.cuda.synchronize()`)

In [1]:
import tensorflow as tf
import time
import numpy as np

# ---- float32 for fair comparison with UniverSeg (PyTorch float32) ----
tf.keras.mixed_precision.set_global_policy('float32')

print(f"TensorFlow  : {tf.__version__}")
print(f"Compute dtype: {tf.keras.mixed_precision.global_policy().compute_dtype}")

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPU available: {gpus[0].name}")
else:
    print("No GPU available – running on CPU.")

TensorFlow  : 2.19.0
Compute dtype: float32
GPU available: /physical_device:GPU:0


In [3]:
from tensorflow.keras import layers
from tensorflow.keras.layers import (
    SeparableConv2D, Conv2D, BatchNormalization, LeakyReLU,
    Dropout, UpSampling2D, Add, concatenate, Conv2DTranspose
)

H = 128  # Height
W = 128  # Width


def build_prompt_unet(height=H, width=W):
    """Build the PromptUNet architecture (identical to training notebook)."""
    inputs = [
        tf.keras.Input(shape=(height, width, 1), name='image'),
        tf.keras.Input(shape=(height, width, 2), name='prompt')
    ]

    image = inputs[0]
    prompt = inputs[1]

    prompt_skip_connections = []
    skip_connections = []

    def conv_block(inp, filters, padding='same', activation='leaky_relu',
                   dropout_rate=0.1, **kwargs):
        inp = Conv2D(filters, (3, 3), padding=padding, **kwargs)(inp)
        inp = BatchNormalization()(inp)
        inp = LeakyReLU()(inp)
        inp = Dropout(dropout_rate)(inp)
        return inp

    def conv_block_prompt(x, p, filters, padding='same',
                          activation='leaky_relu', dropout_rate=0.1):
        p = conv_block(p, filters)
        x = Conv2D(filters, (3, 3), padding=padding)(x)
        x = Add()([x, p])
        x = Dropout(dropout_rate)(x)
        return x

    def conv_block_up(inp, filters, padding='same', activation='leaky_relu',
                      dropout_rate=0.1, **kwargs):
        inp = BatchNormalization()(inp)
        inp = LeakyReLU()(inp)
        inp = Dropout(dropout_rate)(inp)
        inp = Conv2DTranspose(filters, (3,3), padding=padding, **kwargs)(inp)
        return inp

    def encoder_block(p, filters, padding='same', activation='leaky_relu'):
        p = conv_block(p, filters, padding, activation)
        p = conv_block(p, filters, padding, activation)
        prompt_skip_connections.append(p)
        p = conv_block(p, filters * 2, padding, strides=2)
        return p

    def encoder_block_2(x, p, filters, padding='same'):
        x = conv_block_prompt(x, p, filters)
        skip_connections.append(x)
        x = conv_block(x, filters * 2, padding, strides=2)
        return x

    def decoder_block(inp, concat_layer, filters, padding='same',
                      dropout_rate=0.1):
        x = conv_block_up(inp, filters, padding, strides=2)
        x = conv_block(x, filters, padding, dropout_rate=dropout_rate)
        x = concatenate([x, concat_layer])
        x = conv_block(x, filters, padding, dropout_rate=dropout_rate)
        return x

    # --- Encoding prompt ---
    prompt = SeparableConv2D(32, (3, 3), padding='same')(prompt)
    prompt = encoder_block(prompt, 32)
    prompt = encoder_block(prompt, 64)
    prompt = encoder_block(prompt, 128)
    prompt = encoder_block(prompt, 256)
    prompt = encoder_block(prompt, 512)

    # --- Encoding x (with conditioning on prompt) ---
    x = SeparableConv2D(32, (3, 3), padding='same')(image)
    x = encoder_block_2(x, prompt_skip_connections.pop(0), 32)
    x = encoder_block_2(x, prompt_skip_connections.pop(0), 64)
    x = encoder_block_2(x, prompt_skip_connections.pop(0), 128)
    x = encoder_block_2(x, prompt_skip_connections.pop(0), 256)
    x = encoder_block_2(x, prompt_skip_connections.pop(0), 512)

    # --- Middle part (not in skip connection) ---
    x = conv_block(x, 1024, dropout_rate=0.2)

    # --- Decoding / Upsampling (with skip connections) ---
    x = decoder_block(x, skip_connections.pop(), 512)
    x = decoder_block(x, skip_connections.pop(), 256)
    x = decoder_block(x, skip_connections.pop(), 128)
    x = decoder_block(x, skip_connections.pop(), 64)
    x = decoder_block(x, skip_connections.pop(), 32)

    output = Conv2D(1, 1)(x)
    output = tf.keras.activations.sigmoid(output)

    return tf.keras.Model(inputs=inputs, outputs=output)


print('Building PromptUNet model...')
model = build_prompt_unet()
model.summary()
print(f"\nTotal Parameters: {model.count_params() / 1e6:.2f} M")

Building PromptUNet model...


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ prompt (InputLayer) │ (None, 128, 128,  │          0 │ -                 │
│                     │ 2)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ separable_conv2d_2  │ (None, 128, 128,  │        114 │ prompt[0][0]      │
│ (SeparableConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_31 (Conv2D)  │ (None, 128, 128,  │      9,248 │ separable_conv2d… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        128 │ conv2d_31[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_27      │ (None, 128, 128,  │          0 │ batch_normalizat… │
│ (LeakyReLU)         │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_32          │ (None, 128, 128,  │          0 │ leaky_re_lu_27[0… │
│ (Dropout)           │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_32 (Conv2D)  │ (None, 128, 128,  │      9,248 │ dropout_32[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128,  │        128 │ conv2d_32[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_28      │ (None, 128, 128,  │          0 │ batch_normalizat… │
│ (LeakyReLU)         │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_33          │ (None, 128, 128,  │          0 │ leaky_re_lu_28[0… │
│ (Dropout)           │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_33 (Conv2D)  │ (None, 64, 64,    │     18,496 │ dropout_33[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_33[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_29      │ (None, 64, 64,    │          0 │ batch_normalizat… │
│ (LeakyReLU)         │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_34          │ (None, 64, 64,    │          0 │ leaky_re_lu_29[0… │
│ (Dropout)           │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_34 (Conv2D)  │ (None, 64, 64,    │     36,928 │ dropout_34[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 64,    │        256 │ conv2d_34[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_30      │ (None, 64, 64,    │          0 │ batch_normalizat

 Total params: 45,627,900 (174.06 MB)

 Trainable params: 45,606,076 (173.97 MB)

 Non-trainable params: 21,824 (85.25 KB)


Total Parameters: 45.63 M


In [4]:
# Volume simulation parameters
NUM_SLICES = 128   # Number of 2D slices in a simulated volume
B = 1              # Batch size (1 slice at a time, as in real inference)

print(f"Input shape (Image):  (Batch={B}, H={H}, W={W}, C=1)")
print(f"Input shape (Prompt): (Batch={B}, H={H}, W={W}, C=2)")
print(f"Volume simulation:    {NUM_SLICES} slices of {H}x{W}")

# Dummy data for warm-up
dummy_image  = tf.random.normal((B, H, W, 1))
dummy_prompt = tf.random.normal((B, H, W, 2))

Input shape (Image):  (Batch=1, H=128, W=128, C=1)
Input shape (Prompt): (Batch=1, H=128, W=128, C=2)
Volume simulation:    128 slices of 128x128


## Model Complexity (GFLOPs)

In [5]:
# --- Attempt FLOPs calculation via TensorFlow Profiler ---
try:
    from tensorflow.python.profiler.model_analyzer import profile as tf_profile
    from tensorflow.python.profiler.option_builder import ProfileOptionBuilder

    @tf.function(input_signature=[
        tf.TensorSpec(shape=(1, H, W, 1), dtype=tf.float32),
        tf.TensorSpec(shape=(1, H, W, 2), dtype=tf.float32),
    ])
    def _profile_forward(image, prompt):
        return model([image, prompt], training=False)

    concrete_fn = _profile_forward.get_concrete_function()

    graph_info = tf_profile(
        concrete_fn.graph,
        options=ProfileOptionBuilder.float_operation()
    )

    total_flops = graph_info.total_float_ops
    # TF profiler counts all float ops (adds + muls), same unit as thop's  MACs*2
    gflops = total_flops / 1e9

    print(f"Parameters : {model.count_params() / 1e6:.2f} M")
    print(f"Total FLOPs: {total_flops:,}")
    print(f"GFLOPs     : {gflops:.2f}")

except Exception as e:
    print(f"Could not compute FLOPs via TF Profiler: {e}")
    print(f"Parameters: {model.count_params() / 1e6:.2f} M")
    print("GFLOPs: N/A (install a profiler or compute manually)")

Instructions for updating:
This API was designed for TensorFlow v1. See https://www.tensorflow.org/guide/migrate for instructions on how to migrate your code to TensorFlow v2.


Parameters : 45.63 M
Total FLOPs: 13,014,122,432
GFLOPs     : 13.01


## Inference Speed (Volume Simulation)

Simulates processing a full volume of `NUM_SLICES` 128×128 2D images.

**Key optimisations for fair comparison with PyTorch/UniverSeg:**
- `@tf.function` compiled inference (no Python dispatch overhead)
- GPU sync after every slice (equivalent to `torch.cuda.synchronize()`)
- Uses `model(x, training=False)` instead of `model.predict(x)`

In [6]:
# ---- Compile the inference function once ----
@tf.function(input_signature=[
    tf.TensorSpec(shape=(1, H, W, 1), dtype=tf.float32),
    tf.TensorSpec(shape=(1, H, W, 2), dtype=tf.float32),
])
def infer(image, prompt):
    return model([image, prompt], training=False)


# ---- Warm-up (trigger tracing + warm the GPU) ----
print('Warming up model (compiling @tf.function)...')
for _ in range(10):
    _ = infer(dummy_image, dummy_prompt)
# Final sync to make sure warm-up is truly finished
_ = tf.reduce_sum(infer(dummy_image, dummy_prompt)).numpy()
print('Warm-up complete.\n')


# ---- Generate random volume data ----
print(f'Generating {NUM_SLICES} random slices...')
volume_images  = [tf.random.normal((B, H, W, 1)) for _ in range(NUM_SLICES)]
volume_prompts = [tf.random.normal((B, H, W, 2)) for _ in range(NUM_SLICES)]


# ---- Timed volume inference ----
print(f'Running volume inference ({NUM_SLICES} slices)...\n')

start_time = time.perf_counter()

for i in range(NUM_SLICES):
    result = infer(volume_images[i], volume_prompts[i])
    # GPU sync after every slice – equivalent to torch.cuda.synchronize()
    _ = tf.reduce_sum(result).numpy()

end_time = time.perf_counter()

total_volume_time = end_time - start_time
avg_slice_time    = total_volume_time / NUM_SLICES
throughput         = NUM_SLICES / total_volume_time

print(f'--- Volume Inference Results ({NUM_SLICES} slices of {H}x{W}) ---')
print(f'Total volume inference time : {total_volume_time * 1000:.2f} ms')
print(f'Average per-slice time      : {avg_slice_time * 1000:.2f} ms')
print(f'Throughput                  : {throughput:.2f} slices/sec')

Warming up model (compiling @tf.function)...
Warm-up complete.

Generating 128 random slices...
Running volume inference (128 slices)...

--- Volume Inference Results (128 slices of 128x128) ---
Total volume inference time : 1007.75 ms
Average per-slice time      : 7.87 ms
Throughput                  : 127.02 slices/sec


## GPU Memory Usage

In [7]:
if gpus:
    try:
        # Reset peak‐memory tracking
        tf.config.experimental.reset_memory_stats('GPU:0')
    except Exception:
        pass   # API may not be available in older TF versions

    # Run one full volume pass to capture peak memory
    for i in range(NUM_SLICES):
        result = infer(volume_images[i], volume_prompts[i])
        _ = tf.reduce_sum(result).numpy()

    try:
        mem_info = tf.config.experimental.get_memory_info('GPU:0')
        peak_bytes = mem_info['peak']
    except Exception:
        # Fallback: use nvidia-smi style query (less precise)
        peak_bytes = tf.config.experimental.get_memory_usage('GPU:0'.name)

    peak_mb = peak_bytes / (1024 ** 2)
    peak_gb = peak_bytes / (1024 ** 3)
    print(f'Peak GPU Memory Allocated: {peak_mb:.2f} MB ({peak_gb:.3f} GB)')
else:
    print('GPU Memory Usage: N/A (running on CPU)')

Peak GPU Memory Allocated: 483.39 MB (0.472 GB)
